# 실습 — SMOTE와 앙상블을 이온 밀 데이터에 적용하기

| 항목 | 내용 |
|---|---|
| 목표 | 수업에서 배운 **SMOTE**(소수 클래스 늘리기)와 **앙상블**(여러 모델 합치기)이 이 데이터에서 효과가 있는지 직접 확인한다 |
| 문제 | **압력 저하**가 24시간 안에 오는지 맞히는 이진 분류 (**지도 학습**, 정답 = TTF ≤ 24시간) |
| 데이터 | `data/processed/step_s1/` (셔터 1 스텝 구간, 20대) |
| 지표 | 정확도·재현율·정밀도·F1, PR-AUC |

**진행 순서**

| 단계 | 내용 |
|---|---|
| 0 | 준비 (라이브러리, 경로) |
| 1 | 데이터 불러오기 |
| 2 | 시간 순서로 학습/평가 나누기 |
| 3 | 정답(라벨) 만들기 |
| 4 | 입력(피처) 만들기 |
| 5 | 평가 함수 |
| 6 | 기준 모델: 로지스틱 회귀 |
| 7 | 불균형 처리: 가중치 vs **SMOTE** |
| 8 | 트리 모델: 랜덤 포레스트, XGBoost |
| 9 | **앙상블**: 소프트 보팅, 스태킹 |
| 10 | 공정한 비교: 재현율을 맞추고 정밀도 보기 |
| 11 | 직접 해볼 것 |

위에서부터 한 셀씩 실행한다. 전체 실행은 약 3분이다.

## 0. 준비

In [1]:
# ── 기본 도구 ─────────────────────────────────────────────────
from pathlib import Path        # 파일 경로 다루기
import time                     # 학습 시간 재기
import numpy as np              # 배열 계산
import pandas as pd             # 표(데이터프레임) 다루기

# ── scikit-learn: 전처리·모델·지표 ─────────────────────────────────
from sklearn.preprocessing import StandardScaler       # 입력을 평균 0, 표준편차 1로 맞춤 (로지스틱 회귀에 필요)
from sklearn.pipeline import make_pipeline             # 전처리 → 모델을 한 덩어리로 묶음
from sklearn.linear_model import LogisticRegression    # 로지스틱 회귀 (기준 모델)
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier   # 랜덤 포레스트, 앙상블 2종
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, average_precision_score   # 평가 지표

# ── imbalanced-learn: SMOTE ─────────────────────────────────────
from imblearn.over_sampling import SMOTE                            # 소수 클래스(정답 1) 합성
from imblearn.pipeline import make_pipeline as make_imb_pipeline   # SMOTE를 학습할 때만 적용하는 파이프라인

# ── XGBoost ──────────────────────────────────────────────────────
from xgboost import XGBClassifier               # 그래디언트 부스팅 트리 모델

SEED = 42                      # 무작위 요소(SMOTE, 트리)의 시작값. 같으면 매번 같은 결과
HOUR, DAY = 3600, 86400        # 시간 단위(초). 데이터의 시각은 초 단위다

# 노트북을 프로젝트 루트나 models/ 어디서 열어도 데이터 폴더를 찾는다
cwd = Path.cwd().resolve()                                                                    # 지금 노트북이 열린 폴더
ROOT = next(path for path in (cwd, cwd.parent) if (path / 'data/processed/step_s1').exists())   # 지금 폴더 또는 한 단계 위
PROC = ROOT / 'data/processed'                                                                # 가공 데이터 폴더

PRESSURE, FLOW = 'FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE'     # 냉각 압력, 냉각 유량
DROP_FAULT = 'FlowCool Pressure Dropped Below Limit'        # 압력 저하
DROP_TTF = 'TTF_' + DROP_FAULT                              # 압력 저하까지 남은 시간(초) 열

## 1. 데이터 불러오기

- **스텝 구간 데이터**: 한 행 = 셔터 1 상태로 같은 레시피 스텝이 이어진 구간 하나. 센서는 구간 안 평균값을 쓴다.
- **고장 기록**: 같은 장비·시각·종류로 두 번 찍힌 기록은 하나로 본다.

In [2]:
# 필요한 열만 읽는다 (장비, 시각, 레시피·스텝, 정답 TTF, 센서 2개의 구간 평균)
load_cols = ['Tool', 'time_start', 'time_end', 'recipe', 'recipe_step', DROP_TTF, PRESSURE + '_mean', FLOW + '_mean']

# 장비마다 파일이 하나씩 있다(20개) → 모두 읽어서 위아래로 붙인다
segments = pd.concat([pd.read_parquet(path, columns=load_cols) for path in sorted((PROC / 'step_s1').glob('*.parquet'))],
                     ignore_index=True)
# 장비별 시간순으로 정렬 (최근 평균·고장 이력 계산에 필요)
segments = segments.sort_values(['Tool', 'time_start']).reset_index(drop=True)

# 고장 기록 읽기
faults = pd.read_csv(PROC / 'faults.csv')
faults['Tool'] = faults.Tool.str[2:] + '_' + faults.Tool.str[:2]                      # 01M01 → M01_01 (데이터와 이름 맞추기)
faults['fault_type'] = faults.fault_name.str.replace(' [NoWaferID]', '', regex=False)   # 같은 고장의 표기 차이 통일
# 같은 장비·같은 시각·같은 종류로 두 번 찍힌 기록은 하나만 남기고, 장비별 시간순 정렬
faults = faults.drop_duplicates(['Tool', 'time', 'fault_type']).sort_values(['Tool', 'time']).reset_index(drop=True)

print(f'스텝 구간 {len(segments):,}행 · 장비 {segments.Tool.nunique()}대 · 고장 기록 {len(faults):,}건')
segments.head()   # 앞 5행 확인

스텝 구간 526,268행 · 장비 20대 · 고장 기록 1,211건


,Tool,time_start,time_end,recipe,recipe_step,TTF_FlowCool Pressure Dropped Below Limit,FLOWCOOLPRESSURE_mean,FLOWCOOLFLOWRATE_mean
0,M01_01,3283542,3283842,3,3.0,3249342.0,0.424550,0.686933
1,M01_01,3284286,3284586,3,3.0,3248598.0,0.420978,0.687450
2,M01_01,3285074,3285970,2,3.0,3247214.0,0.428303,0.687351
3,M01_01,3287330,3287686,1,3.0,3245498.0,0.410633,0.686505
4,M01_01,3288170,3288310,2,3.0,3244874.0,0.429895,0.687712


## 2. 시간 순서로 학습/평가 나누기

장비마다 기록 기간의 **앞 75%는 학습, 뒤 25%는 평가**로 쓰고, 그 사이 하루는 비운다.

무작위로 섞어 나누면(`train_test_split`) 같은 고장 직전의 비슷한 행이 학습과 평가에 동시에 들어가 점수가 부풀려진다. 실제로는 **과거로 학습해서 미래를 예측**하므로 시간 순서로 나눈다.

In [3]:
tool_first = segments.groupby('Tool').time_start.min()     # 장비별 기록 시작
tool_last = segments.groupby('Tool').time_end.max()        # 장비별 기록 끝
test_start = tool_first + (tool_last - tool_first) * 0.75  # 평가 시작 시각

row_test_start = segments.Tool.map(test_start)
segments['part'] = np.select([segments.time_end < row_test_start - DAY,      # 평가 시작 하루 전까지 → 학습
                              segments.time_start >= row_test_start],        # 평가 시작 이후 → 평가
                             ['train', 'test'], default='gap')               # 사이 하루 → 사용 안 함
segments.part.value_counts()

part
train    398281
test     126754
gap        1233
Name: count, dtype: int64

## 3. 정답(라벨) 만들기

- **정답 1** = 이 구간 끝에서 24시간 안에 압력 저하가 온다 (TTF ≤ 24시간).
- TTF가 비어 있으면 "기록 기간 안에 고장 없음"이다. 그런데 **기록 끝 24시간 안**이면 그 뒤에 고장이 났는지 알 수 없으므로 뺀다.

In [4]:
ttf_drop = segments[DROP_TTF]                         # 압력 저하까지 남은 시간(초). 결측 = 기간 안에 고장 없음
segments['label'] = (ttf_drop <= DAY).astype(int)    # 24시간(86,400초) 이하면 1, 아니면 0 (결측도 0)

# 판단 불가: 기록 끝까지 24시간이 안 남았는데 TTF가 결측 → 기록이 끝난 뒤 고장이 났을 수도 있음
is_unknown = ((segments.Tool.map(tool_last) - segments.time_end) < DAY) & ttf_drop.isna()
segments = segments[~is_unknown].reset_index(drop=True)
print(f'판단 불가로 뺀 행: {is_unknown.sum():,}')

# 구간별 행 수, 정답 1 개수, 정답 1 비율
segments.groupby('part').label.agg(['size', 'sum', 'mean']).rename(
    columns={'size': '행 수', 'sum': '정답 1', 'mean': '정답 1 비율'}).round(4)

판단 불가로 뺀 행: 1,257


,행 수,정답 1,정답 1 비율
part,,,
gap,1233,9,0.0073
test,125497,4503,0.0359
train,398281,26828,0.0674


정답 1이 학습 구간 약 7%, 평가 구간 약 4%뿐인 **불균형 데이터**다. 그래서 정확도만 보면 안 된다(전부 0이라고 해도 정확도 96%).

## 4. 입력(피처) 만들기

모두 **그 시점까지 알 수 있는 값**만 쓴다.

### 4-1. z값 — 같은 공정 단계의 평소 값과 비교

센서 값은 레시피·스텝마다 원래 다르다. 그래서 같은 **장비·레시피·스텝**의 평소 값과 비교한 z값을 쓴다.
- 평소 = 가장 가까운 고장 기록에서 24시간 넘게 떨어진 구간
- 평균·표준편차는 **학습 구간 평소 행에서만** 구한다(평가 구간 정보가 새지 않게)
- 평소 구간이 10개 미만인 조합은 비운다

In [5]:
def nearest_fault_distance(frame):
    # 구간마다 가장 가까운 고장 기록(앞 또는 뒤)까지의 시간(초)
    distance = np.full(len(frame), np.inf)
    for tool, row_index in frame.groupby('Tool').indices.items():
        fault_times = faults.loc[faults.Tool == tool, 'time'].to_numpy()
        if len(fault_times) == 0:
            continue
        starts, ends = frame.time_start.to_numpy()[row_index], frame.time_end.to_numpy()[row_index]
        prev_pos = np.searchsorted(fault_times, starts)          # 구간 시작 앞의 고장
        dist_prev = np.where(prev_pos > 0, starts - fault_times[np.maximum(prev_pos - 1, 0)], np.inf)
        next_pos = np.searchsorted(fault_times, ends)            # 구간 끝 뒤의 고장
        dist_next = np.where(next_pos < len(fault_times), fault_times[np.minimum(next_pos, len(fault_times) - 1)] - ends, np.inf)
        distance[row_index] = np.minimum(dist_prev, dist_next)
    return distance

is_normal = nearest_fault_distance(segments) > DAY
context_key = (segments.Tool.astype(str) + '|' + segments.recipe.astype(str) + '|'
               + segments.recipe_step.fillna(-1).astype(str))            # 장비·레시피·스텝 조합
baseline_rows = (segments.part == 'train') & is_normal

for sensor in [PRESSURE, FLOW]:
    stats = segments[baseline_rows].groupby(context_key[baseline_rows])[sensor + '_mean'].agg(['mean', 'std', 'size'])
    stats = stats[(stats['size'] >= 10) & (stats['std'] > 1e-6)]          # 평소 구간 10개 이상, 표준편차 0 아님
    z_value = (segments[sensor + '_mean'] - context_key.map(stats['mean'])) / context_key.map(stats['std'])
    segments['z_' + sensor] = z_value.clip(-10, 10)                     # 극단값은 ±10에서 자름

segments[['z_' + PRESSURE, 'z_' + FLOW]].describe().round(2)

,z_FLOWCOOLPRESSURE,z_FLOWCOOLFLOWRATE
count,517722.00,517722.00
mean,-0.01,-0.04
std,1.24,1.09
min,-10.00,-10.00
25%,-0.76,-0.32
50%,-0.06,-0.03
75%,0.66,0.24
max,10.00,10.00


### 4-2. 최근 평균과 변화 — 순간값보다 흐름

EDA에서 고장 전 차이는 직전 순간보다 **며칠에 걸친 수준 차이**로 보였다. 장비별로 지난 6시간·24시간·7일 z값 평균을 만들고, **24시간 − 7일**(최근 변화)을 더한다.

In [6]:
rolling_parts = []
for tool, tool_frame in segments.groupby('Tool'):
    by_time = tool_frame.set_index(pd.to_datetime(tool_frame.time_start, unit='s'))   # 시간 기준 창을 쓰려고 시각을 인덱스로
    part = pd.DataFrame(index=tool_frame.index)
    for sensor, short in [(PRESSURE, 'p'), (FLOW, 'f')]:
        z_values = by_time['z_' + sensor]
        for window in ['6h', '24h', '7D']:
            part[f'{short}_r{window}'] = z_values.rolling(window, min_periods=1).mean().to_numpy()   # 지난 window 평균
        part[f'{short}_chg24'] = part[f'{short}_r24h'] - part[f'{short}_r7D']                       # 최근 변화
    rolling_parts.append(part)
rolling_features = pd.concat(rolling_parts).sort_index()
rolling_features.head()

,p_r6h,p_r24h,p_r7D,p_chg24,f_r6h,f_r24h,f_r7D,f_chg24
0,2.328897,2.328897,2.328897,0.0,-0.416274,-0.416274,-0.416274,0.0
1,2.248513,2.248513,2.248513,0.0,-0.010348,-0.010348,-0.010348,0.0
2,2.358371,2.358371,2.358371,0.0,0.064757,0.064757,0.064757,0.0
3,2.355691,2.355691,2.355691,0.0,0.042479,0.042479,0.042479,0.0
4,2.414560,2.414560,2.414560,0.0,0.181228,0.181228,0.181228,0.0


### 4-3. 고장 이력

고장은 한 번 나면 곧 다시 나는 경우가 많다(EDA: 같은 고장의 절반이 24시간 안에 재기록). 그래서 아래 세 가지를 넣는다.
- 마지막 고장(종류 무관) 이후 시간
- 마지막 압력 저하 이후 시간
- 지난 7일 고장 수

시간은 로그를 씌워 큰 값의 영향을 줄인다. 고장이 한 번도 없으면 1년으로 둔다.

In [7]:
NO_FAULT_HOURS = 24 * 365          # 이전 고장이 없을 때 쓰는 값(1년)

def hours_since_last(event_times, starts):
    # starts 시각마다 직전 event_times 이후 몇 시간 지났는지 (없으면 1년)
    if len(event_times) == 0:
        return np.full(len(starts), NO_FAULT_HOURS, dtype=float)
    prev_pos = np.searchsorted(event_times, starts)
    hours = np.where(prev_pos > 0, (starts - event_times[np.maximum(prev_pos - 1, 0)]) / HOUR, NO_FAULT_HOURS)
    return np.minimum(hours, NO_FAULT_HOURS)

history_parts = []
for tool, tool_frame in segments.groupby('Tool'):
    tool_faults = faults[faults.Tool == tool]
    any_times = tool_faults.time.to_numpy()
    drop_times = tool_faults.loc[tool_faults.fault_type == DROP_FAULT, 'time'].to_numpy()
    starts = tool_frame.time_start.to_numpy()
    history_parts.append(pd.DataFrame({
        'since_any': np.log1p(hours_since_last(any_times, starts)),
        'since_drop': np.log1p(hours_since_last(drop_times, starts)),
        'n7d_any': np.searchsorted(any_times, starts) - np.searchsorted(any_times, starts - 7 * DAY),
    }, index=tool_frame.index))
history_features = pd.concat(history_parts).sort_index()
history_features.describe().round(2)

,since_any,since_drop,n7d_any
count,525011.00,525011.00,525011.00
mean,6.26,6.98,0.90
std,1.94,1.98,2.28
min,0.00,0.00,0.00
25%,4.98,5.81,0.00
50%,6.40,7.30,0.00
75%,7.73,9.08,1.00
max,9.08,9.08,27.00


### 4-4. 입력 합치고 학습/평가로 나누기

In [8]:
# 최근 평균(8개) + 고장 이력(3개)을 옆으로 붙이고, 지금 구간의 z값 2개를 더한다 → 입력 13개
features = pd.concat([rolling_features, history_features], axis=1)
features['p_z'] = segments['z_' + PRESSURE]          # 지금 구간의 냉각 압력 z
features['f_z'] = segments['z_' + FLOW]              # 지금 구간의 냉각 유량 z
features = features.fillna(0)                        # z가 없는 행(드문 공정 조합)은 평소(0)로 채움

# 2장에서 정한 구간 이름으로 학습/평가 행을 고른다 (gap 행은 쓰지 않음)
is_train, is_test = segments.part == 'train', segments.part == 'test'
X_train, y_train = features[is_train], segments.label[is_train]   # X = 입력, y = 정답
X_test, y_test = features[is_test], segments.label[is_test]
print(f'입력 {features.shape[1]}개: {list(features.columns)}')
print(f'학습 {len(X_train):,}행 (정답 1 {y_train.mean():.2%}) · 평가 {len(X_test):,}행 (정답 1 {y_test.mean():.2%})')

입력 13개: ['p_r6h', 'p_r24h', 'p_r7D', 'p_chg24', 'f_r6h', 'f_r24h', 'f_r7D', 'f_chg24', 'since_any', 'since_drop', 'n7d_any', 'p_z', 'f_z']
학습 398,281행 (정답 1 6.74%) · 평가 125,497행 (정답 1 3.59%)


## 5. 평가 함수

| 지표 | 뜻 | 이 문제에서 |
|---|---|---|
| 정확도 | 전체 중 맞힌 비율 | 정답 1이 4%라 "전부 0"도 96%. **참고만** |
| 재현율 | 실제 고장 중 잡은 비율 | 놓침(FN)이 비싸므로 **가장 중요** |
| 정밀도 | 고장이라고 한 것 중 실제 고장 비율 | 너무 낮으면 헛걸음(챔버 점검)이 많다 |
| F1 | 재현율과 정밀도의 조화 평균 | |
| PR-AUC | 기준을 바꿔 가며 본 정밀도-재현율 곡선 아래 넓이 | **기준과 무관**한 모델 비교. 무작위면 정답 1 비율(약 0.036) |

In [9]:
results = []          # 모델별 결과를 모아 둘 곳
probabilities = {}    # 모델별 평가 데이터 예측 확률

def evaluate(name, model, threshold=0.5):
    # 학습 → 평가 데이터 확률 예측 → threshold 기준 지표 계산
    timer = time.time()
    model.fit(X_train, y_train)                          # 학습 데이터로만 학습
    probability = model.predict_proba(X_test)[:, 1]      # 평가 데이터의 "정답 1일 확률" (두 번째 열)
    predicted = (probability >= threshold).astype(int)   # 확률이 기준 이상이면 1로 예측
    row = {'모델': name, '정확도': accuracy_score(y_test, predicted),
           '재현율': recall_score(y_test, predicted, zero_division=0),
           '정밀도': precision_score(y_test, predicted, zero_division=0),
           'F1': f1_score(y_test, predicted, zero_division=0),
           'PR-AUC': average_precision_score(y_test, probability), '학습(초)': round(time.time() - timer)}
    results.append(row)                  # 9장에서 전체 표로 모아 본다
    probabilities[name] = probability    # 10장에서 기준을 다시 맞출 때 쓴다
    return pd.DataFrame([row]).set_index('모델').round(3)   # 이 모델 결과만 한 줄로 보여 줌

## 6. 기준 모델 — 로지스틱 회귀 (불균형 처리 없음)

입력 크기가 서로 달라 `StandardScaler`로 맞춘 뒤 학습한다.

In [10]:
# 표준화 → 로지스틱 회귀. max_iter = 학습 반복 최대 횟수 (작으면 수렴 경고가 날 수 있음)
baseline_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
evaluate('로지스틱 회귀 (처리 없음)', baseline_model)

,정확도,재현율,정밀도,F1,PR-AUC,학습(초)
모델,,,,,,
로지스틱 회귀 (처리 없음),0.968,0.23,0.65,0.34,0.347,1


정확도는 높지만 **재현율이 낮다.** 정답 1이 적어서 모델이 대부분 0이라고 답하기 때문이다. 불균형을 처리해 본다.

## 7. 불균형 처리 — 가중치 vs SMOTE

| 방법 | 하는 일 |
|---|---|
| **가중치** (`class_weight='balanced'`) | 정답 1을 틀리면 더 크게 벌점. 데이터는 그대로 |
| **SMOTE** | 정답 1 행 사이를 이어 **가짜 정답 1 행을 만들어** 비율을 맞춤 |

### 7-1. 가중치

In [11]:
# class_weight='balanced': 정답 1이 적은 만큼(약 14배) 정답 1을 틀릴 때 벌점을 크게 준다
weighted_model = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=2000))
evaluate('로지스틱 회귀 + 가중치', weighted_model)

,정확도,재현율,정밀도,F1,PR-AUC,학습(초)
모델,,,,,,
로지스틱 회귀 + 가중치,0.82,0.736,0.134,0.227,0.387,1


### 7-2. SMOTE가 데이터를 어떻게 바꾸는지 먼저 보기

`sampling_strategy=0.5` = 정답 1을 정답 0의 절반까지 늘린다.

In [12]:
# SMOTE는 가까운 정답 1 행끼리의 "사이"에 새 행을 만든다 → 거리를 쓰므로 표준화한 입력에 적용
smote = SMOTE(sampling_strategy=0.5, random_state=SEED)
X_resampled, y_resampled = smote.fit_resample(StandardScaler().fit_transform(X_train), y_train)   # 확인용 (모델에는 안 씀)
# SMOTE 전후 정답 0·1 개수 비교: 정답 0은 그대로, 정답 1만 늘어난다
pd.DataFrame({'SMOTE 전': y_train.value_counts(), 'SMOTE 후': pd.Series(y_resampled).value_counts()}).rename_axis('정답')

,SMOTE 전,SMOTE 후
정답,,
0,371453,371453
1,26828,185726


### 7-3. SMOTE + 로지스틱 회귀

**주의**: SMOTE는 **학습 데이터에만** 적용해야 한다. 평가 데이터에 가짜 행이 들어가면 점수가 거짓이 된다. `imblearn` 파이프라인에 넣으면 `fit`할 때만 SMOTE가 동작하고 `predict`할 때는 건너뛴다.

In [13]:
# 표준화 → SMOTE(학습 때만) → 로지스틱 회귀. 가중치는 쓰지 않는다 (SMOTE와 효과를 따로 보려고)
smote_model = make_imb_pipeline(StandardScaler(), SMOTE(sampling_strategy=0.5, random_state=SEED),
                                LogisticRegression(max_iter=2000))
evaluate('로지스틱 회귀 + SMOTE', smote_model)

,정확도,재현율,정밀도,F1,PR-AUC,학습(초)
모델,,,,,,
로지스틱 회귀 + SMOTE,0.888,0.688,0.197,0.306,0.379,2


## 8. 트리 모델 — 랜덤 포레스트, XGBoost

트리 모델은 크기 맞추기(스케일링)가 필요 없다. XGBoost의 가중치는 `scale_pos_weight = 정답 0 수 / 정답 1 수`로 준다.

In [14]:
positive_weight = (y_train == 0).sum() / (y_train == 1).sum()   # 정답 0이 정답 1의 몇 배인지
print(f'scale_pos_weight = {positive_weight:.1f}')

def make_forest():
    # 랜덤 포레스트: 나무 200그루의 투표
    return RandomForestClassifier(n_estimators=200,          # 나무 수
                                  max_depth=10,              # 나무 깊이 제한 (너무 깊으면 과적합)
                                  min_samples_leaf=50,       # 잎 하나에 최소 50행 (잡음에 덜 흔들림)
                                  class_weight='balanced',   # 불균형 가중치
                                  n_jobs=-1, random_state=SEED)   # n_jobs=-1: CPU 코어 모두 사용

def make_xgboost(**options):
    # XGBoost: 앞 나무가 틀린 부분을 다음 나무가 고쳐 가며 300그루를 쌓는다
    return XGBClassifier(n_estimators=300,      # 나무 수
                         max_depth=5,           # 나무 깊이
                         learning_rate=0.05,    # 나무 하나가 고치는 정도 (작을수록 천천히, 안정적)
                         subsample=0.8,         # 나무마다 행의 80%만 사용 (과적합 방지)
                         colsample_bytree=0.8,  # 나무마다 입력의 80%만 사용
                         n_jobs=-1, random_state=SEED, eval_metric='logloss', **options)

display(evaluate('랜덤 포레스트 + 가중치', make_forest()))
display(evaluate('XGBoost + 가중치', make_xgboost(scale_pos_weight=positive_weight)))   # 정답 1에 약 14배 가중치
# 트리 모델은 표준화가 필요 없어서 SMOTE → XGBoost만 묶는다
display(evaluate('XGBoost + SMOTE', make_imb_pipeline(SMOTE(sampling_strategy=0.5, random_state=SEED), make_xgboost())))

scale_pos_weight = 13.8


,정확도,재현율,정밀도,F1,PR-AUC,학습(초)
모델,,,,,,
랜덤 포레스트 + 가중치,0.88,0.63,0.175,0.274,0.333,70


,정확도,재현율,정밀도,F1,PR-AUC,학습(초)
모델,,,,,,
XGBoost + 가중치,0.91,0.483,0.195,0.278,0.212,10


,정확도,재현율,정밀도,F1,PR-AUC,학습(초)
모델,,,,,,
XGBoost + SMOTE,0.921,0.408,0.202,0.27,0.197,42


## 9. 앙상블 — 여러 모델 합치기

| 방법 | 하는 일 |
|---|---|
| **소프트 보팅** | 세 모델(LR·RF·XGB)의 확률을 평균 |
| **스태킹** | 세 모델의 예측을 입력으로 받아 최종 모델(로지스틱 회귀)이 다시 판단. 학습 데이터를 3조각으로 나눠(cv=3) 예측을 만든다 |

In [15]:
def base_models():
    # 앙상블에 들어갈 세 모델 (매번 새로 만든다)
    return [('lr', make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=2000))),
            ('rf', make_forest()),
            ('xgb', make_xgboost(scale_pos_weight=positive_weight))]

# 소프트 보팅: 세 모델의 "정답 1일 확률"을 평균 (voting='hard'면 0/1 다수결)
display(evaluate('앙상블: 소프트 보팅', VotingClassifier(base_models(), voting='soft')))
# 스태킹: 세 모델의 예측 확률 3개를 입력으로 최종 로지스틱 회귀가 다시 판단
#   cv=3: 학습 데이터를 3조각으로 나눠, 각 조각의 예측은 나머지 2조각으로 학습한 모델이 만든다 (자기 답 베끼기 방지)
display(evaluate('앙상블: 스태킹', StackingClassifier(base_models(), cv=3,
                                                 final_estimator=LogisticRegression(class_weight='balanced', max_iter=2000))))

,정확도,재현율,정밀도,F1,PR-AUC,학습(초)
모델,,,,,,
앙상블: 소프트 보팅,0.905,0.646,0.22,0.328,0.345,88


,정확도,재현율,정밀도,F1,PR-AUC,학습(초)
모델,,,,,,
앙상블: 스태킹,0.817,0.755,0.134,0.228,0.277,102


### 기준 0.5로 본 전체 결과

In [16]:
# 지금까지 evaluate()가 모아 둔 결과를 한 표로
result_table = pd.DataFrame(results).set_index('모델')
print(f'평가 정답 1 비율(= 무작위 PR-AUC): {y_test.mean():.3f}')   # PR-AUC가 이보다 높아야 의미가 있다
result_table.round(3)

평가 정답 1 비율(= 무작위 PR-AUC): 0.036


,정확도,재현율,정밀도,F1,PR-AUC,학습(초)
모델,,,,,,
로지스틱 회귀 (처리 없음),0.968,0.230,0.650,0.340,0.347,1
로지스틱 회귀 + 가중치,0.820,0.736,0.134,0.227,0.387,1
로지스틱 회귀 + SMOTE,0.888,0.688,0.197,0.306,0.379,2
랜덤 포레스트 + 가중치,0.880,0.630,0.175,0.274,0.333,70
XGBoost + 가중치,0.910,0.483,0.195,0.278,0.212,10
XGBoost + SMOTE,0.921,0.408,0.202,0.270,0.197,42
앙상블: 소프트 보팅,0.905,0.646,0.220,0.328,0.345,88
앙상블: 스태킹,0.817,0.755,0.134,0.228,0.277,102


이 표로 순위를 매기면 안 된다. 가중치·SMOTE를 쓰면 확률이 전체적으로 위로 밀려서, **같은 0.5라도 모델마다 의미가 다르다.** 다음 단계에서 조건을 맞춰 비교한다.

## 10. 공정한 비교 — 재현율을 70%로 맞추고 정밀도 보기

이온 밀은 놓침이 비싸다. 그래서 **모든 모델의 재현율을 70%로 똑같이 맞춘 뒤, 정밀도가 높은 모델**(헛걸음이 적은 모델)을 고른다.

실습이라 기준을 평가 데이터에서 맞췄다. 실제 운영이라면 학습 기간 뒤쪽(보정 구간)에서 정해야 한다. 여기서는 순위 비교용이다.

In [17]:
TARGET_RECALL = 0.7     # 모든 모델의 재현율을 이 값으로 맞춘다

matched_rows = []
for name, probability in probabilities.items():
    # 실제 정답 1인 행들의 확률을 작은 순으로 정렬
    positive_scores = np.sort(probability[y_test.to_numpy() == 1])
    # 아래 30% 지점의 확률을 기준으로 → 정답 1의 70%가 이 기준 이상 → 재현율 70%
    threshold = positive_scores[int(len(positive_scores) * (1 - TARGET_RECALL))]
    predicted = (probability >= threshold).astype(int)
    # 재현율은 모두 0.7로 같으니, 정밀도가 높을수록 헛경보가 적은 좋은 모델
    matched_rows.append({'모델': name, '기준': round(threshold, 3),
                         '재현율': recall_score(y_test, predicted), '정밀도': precision_score(y_test, predicted),
                         'F1': f1_score(y_test, predicted), 'PR-AUC': average_precision_score(y_test, probability)})
pd.DataFrame(matched_rows).set_index('모델').sort_values('정밀도', ascending=False).round(3)

,기준,재현율,정밀도,F1,PR-AUC
모델,,,,,
로지스틱 회귀 + 가중치,0.638,0.7,0.192,0.301,0.387
앙상블: 소프트 보팅,0.430,0.7,0.183,0.290,0.345
로지스틱 회귀 + SMOTE,0.464,0.7,0.180,0.286,0.379
앙상블: 스태킹,0.644,0.7,0.177,0.283,0.277
로지스틱 회귀 (처리 없음),0.099,0.7,0.168,0.271,0.347
XGBoost + 가중치,0.204,0.7,0.115,0.197,0.212
랜덤 포레스트 + 가중치,0.379,0.7,0.109,0.189,0.333
XGBoost + SMOTE,0.145,0.7,0.098,0.172,0.197


## 해석

모두 **지도 학습**, 정답 = 24시간 안 압력 저하. 평가 양성 비율 3.6%(학습 구간 6.7%).

**재현율 70%에 맞췄을 때 (공정한 비교)**

| 순위 | 모델 | 정밀도 | PR-AUC |
|---|---|---|---|
| 1 | 로지스틱 회귀 + 가중치 | **0.192** | **0.387** |
| 2 | 앙상블: 소프트 보팅 | 0.183 | 0.345 |
| 3 | 로지스틱 회귀 + SMOTE | 0.180 | 0.379 |
| 4 | 앙상블: 스태킹 | 0.177 | 0.277 |
| 5 | 로지스틱 회귀 (처리 없음) | 0.168 | 0.347 |
| 6~8 | XGBoost + 가중치 / 랜덤 포레스트 / XGBoost + SMOTE | 0.098~0.115 | 0.197~0.333 |

- **SMOTE: 적용은 되지만 이득이 없다.** 같은 로지스틱 회귀에서 가중치(정밀도 0.192)보다 SMOTE(0.180)가 조금 낮고, XGBoost에서도 SMOTE가 더 낮다(0.115 → 0.098). 둘 다 "양성을 더 중요하게 보라"는 같은 일을 하는데, SMOTE는 가짜 행을 만들어 학습 시간만 늘린다. 이 데이터는 양성이 2.7만 행으로 적지 않아 합성할 필요가 크지 않다.
- **앙상블: 적용은 되지만 단일 로지스틱 회귀를 못 이긴다.** 소프트 보팅 0.183, 스태킹 0.177로 로지스틱 회귀 + 가중치(0.192)보다 낮다. 앙상블은 구성 모델이 각자 쓸 만해야 효과가 있는데, 트리 모델(RF·XGBoost)이 0.10~0.12로 약해 오히려 끌어내렸다.
- **트리 모델이 약한 이유**: 학습 구간과 평가 구간의 고장 빈도가 다르고(6.7% → 3.6%), 장비마다 압력 방향도 다르다(eda_tools). 복잡한 모델은 학습 기간의 패턴을 외워서 다음 기간에 덜 맞는다. 단순한 로지스틱 회귀가 더 잘 버틴다.
- **기준 0.5 표(9장 끝)는 비교에 쓰면 안 된다.** "처리 없음"이 정확도 0.97, 정밀도 0.65로 좋아 보이지만 재현율 0.23이다(고장 77%를 놓침). 가중치·SMOTE는 확률을 위로 밀어 같은 0.5에서 재현율이 오른 것뿐이다. 이 문제는 놓침이 비싸므로 **재현율을 맞춘 뒤 정밀도로** 비교한다.

**결론**: SMOTE와 앙상블 모두 이 데이터에 **적용은 가능하지만 성능 이득은 없다.** 지금처럼 **로지스틱 회귀 + 가중치**를 유지한다. 성능을 올리려면 모델을 바꾸는 것보다 입력(장비별 기준, 최근 대비 변화, 셔터 0 시간 비율 등)을 개선하는 쪽이 낫다.

**한계**: 10장 기준은 실습이라 평가 데이터에서 맞췄다(순위 비교용). 행 단위 지표라 현장 채점(사건 단위, 오경보/주)과는 다르다. 압력 저하 하나만 봤다.

## 11. 직접 해볼 것

위 셀의 숫자를 바꿔 다시 실행해 본다.

| 해볼 것 | 바꿀 곳 | 확인할 것 |
|---|---|---|
| SMOTE 비율 바꾸기 | 7-3의 `sampling_strategy=0.5` → `0.2`, `1.0` | 정밀도가 달라지는가 |
| SMOTE 대신 줄이기 | `from imblearn.under_sampling import RandomUnderSampler`로 정답 0을 줄이기 | 학습 시간과 성능 |
| 재현율 목표 바꾸기 | 10장의 `TARGET_RECALL = 0.7` → `0.5`, `0.9` | 정밀도가 얼마나 떨어지는가 |
| 트리 모델 단순하게 | 8장 `max_depth` 10 → 3, `n_estimators` 줄이기 | 과적합이 줄어 성능이 오르는가 |
| 입력 빼 보기 | 4-4에서 `history_features`를 빼고 실행 | 고장 이력이 얼마나 중요한가 |
| 다른 고장 | 0장의 `DROP_FAULT`를 `'Flowcool leak'`(누설)으로 | 고장 종류마다 결과가 다른가 |
| 무작위로 나누면? | 2장 대신 `train_test_split(..., shuffle=True)` | 점수가 얼마나 부풀려지는가 (왜 시간 순서로 나누는지 확인) |